# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
! pip install -q schedule pytest # установка библиотек, если ещё не

/bin/bash: line 1: pip: command not found


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import re
import json
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
from bs4 import BeautifulSoup


def get_book_data(book_url: str) -> dict:
    """
    Парсит информацию о книге со страницы каталога Books to Scrape.

    Получает HTML-код страницы, извлекает название книги, цену, рейтинг,
    количество экземпляров в наличии, описание и все характеристики
    из таблицы Product Information.

    Args:
        book_url (str): URL-адрес страницы конкретной книги.

    Returns:
        dict: Словарь с данными о книге (название, цена, рейтинг, наличие,
            описание и параметры из таблицы Product Information).
    """
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url)
    response.raise_for_status()
    response.encoding = 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')
    book_data = {}

    # 1. Название книги
    title_tag = soup.find('div', class_='product_main').find('h1')
    book_data['Title'] = title_tag.text.strip() if title_tag else None

    # 2. Цена
    price_tag = soup.find('p', class_='price_color')
    book_data['Price'] = price_tag.text.strip() if price_tag else None

    # 3. Рейтинг (хранится во втором классе тега <p class="star-rating Three">)
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    rating_tag = soup.find('p', class_='star-rating')
    if rating_tag and len(rating_tag.get('class', [])) > 1:
        rating_word = rating_tag['class'][1]
        book_data['Rating'] = rating_map.get(rating_word, rating_word)
    else:
        book_data['Rating'] = None

    # 4. Количество в наличии (из строки вида "In stock (22 available)")
    stock_tag = soup.find('p', class_='instock availability')
    if stock_tag:
        stock_text = stock_tag.text.strip()
        stock_match = re.search(r'\((\d+)\s+available\)', stock_text)
        book_data['Availability'] = (
            int(stock_match.group(1)) if stock_match else stock_text
        )
    else:
        book_data['Availability'] = None

    # 5. Описание (идёт в теге <p> сразу после <div id="product_description">)
    desc_header = soup.find('div', id='product_description')
    if desc_header:
        desc_tag = desc_header.find_next_sibling('p')
        book_data['Description'] = desc_tag.text.strip() if desc_tag else None
    else:
        book_data['Description'] = None

    # 6. Таблица Product Information
    table = soup.find('table', class_='table-striped')
    if table:
        for row in table.find_all('tr'):
            header = row.find('th')
            value = row.find('td')
            if header and value:
                book_data[header.text.strip()] = value.text.strip()

    return book_data
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'Title': 'A Light in the Attic',
 'Price': '£51.77',
 'Rating': 3,
 'Availability': 'In stock (22 available)',
 'Description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place 

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def _create_session() -> requests.Session:
    """Создаёт HTTP-сессию с автоматическими повторами при сетевых сбоях."""
    session = requests.Session()
    retries = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session


def get_book_data(
    book_url: str, session: requests.Session | None = None
) -> dict:
    """
    Парсит информацию о книге со страницы каталога Books to Scrape.

    Args:
        book_url (str): URL-адрес страницы конкретной книги.
        session (requests.Session | None): Опциональная HTTP-сессия для
            переиспользования соединений.

    Returns:
        dict: Словарь с данными о книге.
    """
    requester = session or requests
    response = requester.get(book_url, timeout=15)
    response.raise_for_status()
    response.encoding = 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')
    book_data = {}

    # 1. Название книги
    main_div = soup.find('div', class_='product_main')
    title_tag = main_div.find('h1') if main_div else None
    book_data['Title'] = title_tag.text.strip() if title_tag else None

    # 2. Цена
    price_tag = soup.find('p', class_='price_color')
    book_data['Price'] = price_tag.text.strip() if price_tag else None

    # 3. Рейтинг
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    rating_tag = soup.find('p', class_='star-rating')
    if rating_tag and len(rating_tag.get('class', [])) > 1:
        rating_word = rating_tag['class'][1]
        book_data['Rating'] = rating_map.get(rating_word, rating_word)
    else:
        book_data['Rating'] = None

    # 4. Количество в наличии
    stock_tag = soup.find('p', class_='instock availability')
    if stock_tag:
        stock_text = stock_tag.text.strip()
        stock_match = re.search(r'\((\d+)\s+available\)', stock_text)
        book_data['Availability'] = (
            int(stock_match.group(1)) if stock_match else stock_text
        )
    else:
        book_data['Availability'] = None

    # 5. Описание
    desc_header = soup.find('div', id='product_description')
    if desc_header:
        desc_tag = desc_header.find_next_sibling('p')
        book_data['Description'] = desc_tag.text.strip() if desc_tag else None
    else:
        book_data['Description'] = None

    # 6. Таблица Product Information
    table = soup.find('table', class_='table-striped')
    if table:
        for row in table.find_all('tr'):
            header = row.find('th')
            value = row.find('td')
            if header and value:
                book_data[header.text.strip()] = value.text.strip()

    return book_data


def scrape_books(is_save: bool = False) -> list[dict]:
    """
    Собирает информацию обо всех книгах из каталога Books to Scrape.

    Args:
        is_save (bool): Флаг сохранения результата в файл books_data.txt.
            По умолчанию False.

    Returns:
        list[dict]: Список словарей с данными обо всех книгах.
    """
    all_books = []
    page_num = 1

    with _create_session() as session:
        while True:
            page_url = (
                f'http://books.toscrape.com/catalogue/page-{page_num}.html'
            )
            response = session.get(page_url, timeout=15)

            if response.status_code == 404:
                break

            response.raise_for_status()
            response.encoding = 'utf-8'

            soup = BeautifulSoup(response.text, 'html.parser')
            book_cards = soup.find_all('article', class_='product_pod')

            if not book_cards:
                break

            for card in book_cards:
                h3_tag = card.find('h3')
                link_tag = h3_tag.find('a') if h3_tag else None
                if link_tag and link_tag.get('href'):
                    book_url = urljoin(page_url, link_tag['href'])
                    book_info = get_book_data(book_url, session=session)
                    all_books.append(book_info)

            page_num += 1

    if is_save:
        with open('books_data.txt', 'w', encoding='utf-8') as file:
            json.dump(all_books, file, ensure_ascii=False, indent=4)

    return all_books

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

def run_scheduled_scraping() -> None:
    """
    Запускает сбор данных о книгах и сохраняет результат в файл books_data.txt.
    """
    print('Запуск планового сбора данных...')
    scrape_books(is_save=True)
    print('Сбор данных успешно завершён, файл books_data.txt обновлён.')


schedule.clear()
schedule.every().day.at('22:00').do(run_scheduled_scraping)

try:
    while True:
        schedule.run_pending()
        time.sleep(30)
except KeyboardInterrupt:
    print('Планировщик остановлен пользователем.')
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

Запуск планового сбора данных...
Сбор данных успешно завершён, файл books_data.txt обновлён.
Планировщик остановлен пользователем.


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [11]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest test/test_scraper.py

============================= test session starts ==============================
platform linux -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0
rootdir: /srv/code/mipt/pythoncourse
configfile: pyproject.toml
collected 4 items                                                              

test/test_scraper.py ....                                                [100%]

======================== 4 passed in 184.40s (0:03:04) =========================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: было задано несколько задач на парсинг, гит и тесты. Получились несколько громоздкие решения`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: в основном все было хорошо, немного застрял с таймаутами`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Использовал для дебага hang-а`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: вроде бы все понятно`

**5. Самооценка по критериям задания**:

`Ответ: 9/10 of 10`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: ...`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

